# TensorFlow’s CUDA build did not include my GPU’s compute capability.

- GPU: RTX 5070 (Blackwell) → compute capability 12.0 (sm_120).

- Pip TensorFlow: Built with older cuda_compute_capabilities (e.g. up to sm_90 or so), so it had no prebuilt kernels for sm_120. TensorFlow then either:

    - tried to JIT-compile from PTX (slow, “could take 30 minutes or longer”), or
    - hit runtime errors like CUDA_ERROR_INVALID_HANDLE / “Cannot dlopen some GPU libraries” in WSL2.

- Conda-forge TensorFlow: Built with sm_120 and compute_120 in cuda_compute_capabilities (as in your notebook’s get_build_info()), so it had native kernels for the RTX 5070 and worked.

So the fix was using a TensorFlow build that was compiled with cuda_compute_capabilities that include sm_120 (the conda-forge build), instead of the pip build that didn’t.

# Pytorch on torch_gpu

using `pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130`

In [ ]:
import torch

# Check if PyTorch is installed and print its version
print("PyTorch version:", torch.__version__)

# Print some important PyTorch settings
print("PyTorch built with CUDA:", torch.cuda.is_available())
print("PyTorch CUDA version (compiled):", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())

# Test simple tensor operation
x = torch.tensor([1.0, 2.0, 3.0])
y = torch.tensor([4.0, 5.0, 6.0])
z = x + y
print("x + y =", z)

# Check if CUDA is available and print device info
if torch.cuda.is_available():
    print("CUDA is available!")
    device_count = torch.cuda.device_count()
    print("CUDA device count:", device_count)
    for i in range(device_count):
        print(f"\nDevice {i}:")
        print("  Name:", torch.cuda.get_device_name(i))
        capability = torch.cuda.get_device_capability(i)
        print(
            "  CUDA Capability (sm version):",
            capability,
            f"(sm_{capability[0]}{capability[1]})",
        )
        print("  Memory Allocated:", torch.cuda.memory_allocated(i))
        print("  Memory Cached:", torch.cuda.memory_reserved(i))
    current_device = torch.cuda.current_device()
    print(f"\nCurrent CUDA device: {current_device}")
    # Try moving a tensor to GPU
    x_cuda = x.to("cuda")
    print("Tensor on CUDA:", x_cuda)
else:
    print("CUDA is not available.")

PyTorch version: 2.10.0+cu130
PyTorch built with CUDA: True
PyTorch CUDA version (compiled): 13.0
cuDNN version: 91501
x + y = tensor([5., 7., 9.])
CUDA is available!
CUDA device count: 1

Device 0:
  Name: NVIDIA GeForce RTX 5070 Laptop GPU
  CUDA Capability (sm version): (12, 0) (sm_120)
  Memory Allocated: 0
  Memory Cached: 0

Current CUDA device: 0
Tensor on CUDA: tensor([1., 2., 3.], device='cuda:0')


# TF on tf_condaforge

Using `conda install -c conda-forge tensorflow=2.19.1`

In [1]:
! nvidia-smi

Sat Apr 11 13:49:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.54                 Driver Version: 595.79         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5070 ...    On  |   00000000:02:00.0  On |                  N/A |
| N/A   46C    P5             16W /   57W |     758MiB /   8151MiB |     55%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
! nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Tue_May_27_02:21:03_PDT_2025
Cuda compilation tools, release 12.9, V12.9.86
Build cuda_12.9.r12.9/compiler.36037853_0


In [3]:
import tensorflow as tf

print(tf.__version__)
print(tf.config.list_physical_devices("GPU"))

2026-04-11 13:49:28.730929: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-11 13:49:28.743724: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775929768.757109   49576 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775929768.761328   49576 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775929768.771188   49576 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

2.19.1
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
import tensorflow as tf

print(tf.__version__)
print(tf.config.list_physical_devices("GPU"))

2.19.1
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [5]:
import tensorflow as tf

print(tf.sysconfig.get_build_info())

OrderedDict([('cpu_compiler', '/home/conda/feedstock_root/build_artifacts/tensorflow-split_1769770489445/_build_env/bin/x86_64-conda-linux-gnu-gcc'), ('cuda_compute_capabilities', ['sm_60', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_89', 'sm_90', 'sm_100', 'sm_120', 'compute_120']), ('cuda_version', '12.8'), ('cudnn_version', '9'), ('is_cuda_build', True), ('is_rocm_build', False), ('is_tensorrt_build', False)])


In [6]:
print(tf.sysconfig.get_build_info())

OrderedDict([('cpu_compiler', '/home/conda/feedstock_root/build_artifacts/tensorflow-split_1769770489445/_build_env/bin/x86_64-conda-linux-gnu-gcc'), ('cuda_compute_capabilities', ['sm_60', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_89', 'sm_90', 'sm_100', 'sm_120', 'compute_120']), ('cuda_version', '12.8'), ('cudnn_version', '9'), ('is_cuda_build', True), ('is_rocm_build', False), ('is_tensorrt_build', False)])


In [7]:
# pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130

In [8]:
#      python -c "import tensorflow as tf; print(tf.sysconfig.get_build_info())"


In [9]:
tf.config.list_physical_devices("GPU")

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [10]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(tf.config.experimental.get_device_details(gpus[0]))
else:
    print("No TensorFlow GPU devices (empty list). Skipping get_device_details.")
    print(
        "If nvidia-smi works but this is [], check TF build vs GPU generation "
        "(e.g. pip wheels often cap at sm_90; Blackwell needs a build with sm_120) "
        "and cuDNN/driver messages on the first import cell."
    )
    caps = tf.sysconfig.get_build_info().get("cuda_compute_capabilities")
    print("This binary reports cuda_compute_capabilities:", caps)

{'compute_capability': (12, 0), 'device_name': 'NVIDIA GeForce RTX 5070 Laptop GPU'}


In [11]:
import tensorflow as tf

print("TF:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU devices:", gpus)

# Avoid silently falling back when no GPU is registered; large matmul on CPU can hang the notebook.
device = "/GPU:0" if gpus else "/CPU:0"
print("Running matmul on:", device)
with tf.device(device):
    a = tf.random.normal((1000, 1000))
    b = tf.random.normal((1000, 1000))
    c = tf.matmul(a, b)

print(c)

TF: 2.19.1
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Running matmul on: /GPU:0


I0000 00:00:1775929771.934900   49576 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5199 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5070 Laptop GPU, pci bus id: 0000:02:00.0, compute capability: 12.0


tf.Tensor(
[[-33.241516   54.26529   -34.372787  ...  48.08514    20.16751
   73.45491  ]
 [ 23.620829   49.137943    4.7502174 ...  37.089256  -35.445045
   36.887737 ]
 [-26.088951   17.678633  -53.33481   ...   6.4899573  14.080814
    3.4437814]
 ...
 [ 14.665475  -12.683886   -2.2926462 ... -30.817837    4.33634
   16.812239 ]
 [-26.615257  -32.452053  -12.69456   ...  21.703867   14.4621315
   -1.2067071]
 [  6.197507  -27.563904   17.722391  ...  -3.6335194  16.405352
  -19.648891 ]], shape=(1000, 1000), dtype=float32)
